In [ ]:



import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator





import zipfile, os, cv2, numpy as np
from ultralytics import YOLO
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense


zip_path = r"C:\Users\maria\Downloads\yolo_dataset1.zip"
extract_path = r"C:\Users\maria\Downloads\yolo_dataset1"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted to:", extract_path)
print(os.listdir(extract_path))


model = YOLO("yolov8n.pt")
model.train(data=os.path.join(extract_path, "data.yaml"), epochs=50, imgsz=640)


def build_alphanumeric_model():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(36, activation='softmax')  # 26 letters + 10 digits
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

char_model = build_alphanumeric_model()


def classify_color(image_region):
    hsv = cv2.cvtColor(image_region, cv2.COLOR_BGR2HSV)
    avg_color = np.mean(hsv.reshape(-1, 3), axis=0)
    h, s, v = avg_color
    if h < 10 or h > 160: return "Red"
    elif 35 < h < 85: return "Green"
    elif 90 < h < 130: return "Blue"
    elif s < 30 and v > 200: return "White"
    elif v < 50: return "Black"
    elif 130 < h < 160: return "Purple"
    elif 10 < h < 25: return "Brown"
    elif 20 < h < 35: return "Orange"
    else: return "Other"


def process_image(img_path):
    results = model.predict(source=img_path)
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            shape_crop = r.orig_img[y1:y2, x1:x2]
            shape_color = classify_color(shape_crop)
            char_crop = cv2.resize(shape_crop, (28,28))
            char_gray = cv2.cvtColor(char_crop, cv2.COLOR_BGR2GRAY).reshape(1,28,28,1)
            char_pred = char_model.predict(char_gray)
            char_label = np.argmax(char_pred)
            print(f"Shape: {box.cls}, Character: {char_label}, Shape Color: {shape_color}")


Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\maria\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Files extracted to: C:\Users\maria\Downloads\yolo_dataset1
['data.yaml', 'images', 'labels']